# Технологии обработки больших данных

## Лабораторная работа 3. Построение промышленного ETL/ELT-пайплайна

**Вариант:** 1, Amazon Reviews 2023, категория All_Beauty.

**Сценарий:** рекомендательная система. Раз в сутки нужна витрина по товару: число отзывов, средняя оценка, доля подтверждённых покупок. SLA учебного сценария — данные готовы к утру, сбой не должен удваивать строки.

**Цель:** собрать ELT-пайплайн с идемпотентной загрузкой, очередью битых записей, проверкой схемы, тестами качества и трансформацией в отдельной SQL-модели.

**Почему ELT, а не ETL:** сырой слой дешёвый, а агрегаты для рекомендаций проще пересчитать в хранилище, чем тащить бизнес-правила в код извлечения. Для этой категории объём небольшой, DuckDB это выдерживает. На нескольких категориях тот же SQL уезжает в Spark, оркестрация не меняется.


## Подготовка

Ставим DuckDB, PyArrow и dbt. Если dbt в этой среде не соберётся, модель всё равно выполнится тем же SQL через DuckDB, а код установки не роняет весь блокнот.


In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "pyarrow", "duckdb", "matplotlib"])
dbt_code = subprocess.call([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "dbt-core", "dbt-duckdb"])
ge_code = subprocess.call([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "great_expectations"])
subprocess.call([sys.executable, "-m", "pip", "install", "-q", "numpy>=2"])
print("dbt:", dbt_code, "great_expectations:", ge_code)


dbt: 0 great_expectations: 0


## Проектирование

Пайплайн линейный: скачать JSONL, проверить схему, отложить битые строки, слить чистые по идемпотентному ключу, собрать витрину товара.

```mermaid
flowchart LR
  src[JSONL_All_Beauty] --> extract[Извлечение]
  extract --> schema[Проверка_схемы]
  schema --> good[Чистые_строки]
  schema --> dlq[DLQ]
  good --> merge[MERGE_по_ключу]
  merge --> dbt[SQL_витрина_товара]
```


## Извлечение

Качаем ту же публичную категорию, что и в первой работе, и складываем её в таблицу сырых отзывов.


In [2]:
import gzip
import json
import os
import socket
import time
from pathlib import Path
import urllib.request

import duckdb
import pandas as pd

socket.setdefaulttimeout(60)

try:
    import google.colab  # noqa: F401
    BASE = Path("/content/lab3_amazon")
except ImportError:
    BASE = Path(os.environ.get("TEMP", "/tmp")) / "lab3_amazon"
BASE.mkdir(parents=True, exist_ok=True)

URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz"
GZ_PATH = BASE / "All_Beauty.jsonl.gz"
EXPECTED_GZ = 94441517
DB_PATH = BASE / "warehouse.duckdb"

if not GZ_PATH.exists() or GZ_PATH.stat().st_size != EXPECTED_GZ:
    print("Скачиваем отзывы...")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    got = 0
    with urllib.request.urlopen(req, timeout=60) as response, open(GZ_PATH, "wb") as out:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)
            got += len(chunk)
    print(f"Скачано {got / 1024 ** 2:.1f} МБ")

rows = []
with gzip.open(GZ_PATH, "rt", encoding="utf-8") as src:
    for line in src:
        obj = json.loads(line)
        rows.append({
            "user_id": str(obj.get("user_id") or ""),
            "asin": str(obj.get("asin") or ""),
            "rating": obj.get("rating"),
            "timestamp": obj.get("timestamp"),
            "verified_purchase": bool(obj.get("verified_purchase")),
            "text": str(obj.get("text") or ""),
        })

raw = pd.DataFrame(rows)
raw["rating"] = pd.to_numeric(raw["rating"], errors="coerce")
raw["timestamp"] = pd.to_numeric(raw["timestamp"], errors="coerce")
print("Сырых строк:", len(raw))
raw.head(3)


Сырых строк: 701528


,user_id,asin,rating,timestamp,verified_purchase,text
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,1588687728923,True,This spray is really nice. It smells really go...
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,1588615855070,True,"This product does what I need it to do, I just..."
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B07PNNCSP9,5.0,1589665266052,True,"Smells good, feels great!"


## Схема, битые записи и идемпотентный ключ

Ключ — пользователь, товар и время отзыва. Строки без товара или с оценкой вне 1–5 не попадают в витрину, а уходят в DLQ. Повторная загрузка обновляет уже известный ключ и не добавляет вторую копию.


In [3]:
EXPECTED = {"user_id", "asin", "rating", "timestamp", "verified_purchase", "text"}

def check_schema(frame):
    missing = EXPECTED - set(frame.columns)
    if missing:
        raise ValueError(f"Схема изменилась, нет столбцов: {sorted(missing)}")
    return True

check_schema(raw)

# Искусственный дрейф схемы: лишний обязательный столбец в контракте
drift_caught = False
try:
    check_schema(raw.drop(columns=["asin"]))
except ValueError as exc:
    drift_caught = True
    print("Дрейф пойман:", exc)

# Несколько заведомо битых строк, чтобы очередь DLQ было на чем проверить
bad_demo = pd.DataFrame([
    {"user_id": "demo", "asin": "", "rating": -1, "timestamp": 0, "verified_purchase": False, "text": "битая строка"},
    {"user_id": "demo2", "asin": "B000TEST", "rating": 9, "timestamp": None, "verified_purchase": False, "text": "оценка вне шкалы"},
])
raw = pd.concat([raw, bad_demo], ignore_index=True)
bad_mask = raw["asin"].eq("") | raw["rating"].isna() | ~raw["rating"].between(1, 5) | raw["timestamp"].isna()
dlq = raw.loc[bad_mask].copy()
good = raw.loc[~bad_mask].copy()
good = good.drop_duplicates(["user_id", "asin", "timestamp"])
dlq_path = BASE / "dlq.parquet"
dlq.to_parquet(dlq_path, index=False)
print("В DLQ:", len(dlq), "чистых:", len(good))

if DB_PATH.exists():
    DB_PATH.unlink()
con = duckdb.connect(DB_PATH.as_posix())

def duck(sql):
    return con.execute(sql).fetchall()

duck("""
    CREATE TABLE reviews_clean (
        idempotency_key VARCHAR PRIMARY KEY,
        user_id VARCHAR,
        asin VARCHAR,
        rating DOUBLE,
        timestamp BIGINT,
        verified_purchase BOOLEAN,
        text VARCHAR
    )
""")

def load_good(frame):
    # DuckDB подхватывает датафрейм по имени из глобальной области
    global BATCH
    BATCH = frame
    duck("""
        INSERT INTO reviews_clean
        SELECT
            md5(user_id || '|' || asin || '|' || CAST(timestamp AS VARCHAR)) AS idempotency_key,
            user_id, asin, rating, CAST(timestamp AS BIGINT), verified_purchase, text
        FROM BATCH
        ON CONFLICT (idempotency_key) DO UPDATE SET
            rating = excluded.rating,
            text = excluded.text,
            verified_purchase = excluded.verified_purchase
    """)

load_good(good)
count_first = con.execute("SELECT count(*) FROM reviews_clean").fetchone()[0]
load_good(good)
count_second = con.execute("SELECT count(*) FROM reviews_clean").fetchone()[0]
idempotency_passed = count_first == count_second
print("Строк после первого прогона:", count_first)
print("Строк после второго прогона:", count_second)
print("Идемпотентность:", idempotency_passed)

# Time travel в файловом DuckDB нет как в Delta, поэтому снимок до повторной загрузки храним числом строк
snapshot_rows = count_first
print("Снимок числа строк для отката проверки:", snapshot_rows)


Дрейф пойман: Схема изменилась, нет столбцов: ['asin']


В DLQ: 2 чистых: 694252


Строк после первого прогона: 694252
Строк после второго прогона: 694252
Идемпотентность: True
Снимок числа строк для отката проверки: 694252


## Оркестрация с повторами

Шаги идут по графу. У извлечения три попытки и растущая пауза: первый вызов специально падает, второй проходит. Так проверяется восстановление без Airflow, которого в Colab нет.


In [4]:
calls = {"n": 0}

def extract_flaky():
    calls["n"] += 1
    if calls["n"] == 1:
        raise ConnectionError("источник временно недоступен")
    return len(raw)

def run_step(name, func, retries=3):
    delay = 0.2
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            value = func()
            print(f"{name}: успех с попытки {attempt}")
            return value
        except Exception as exc:
            last_error = exc
            print(f"{name}: попытка {attempt} не удалась ({exc})")
            time.sleep(delay)
            delay *= 2
    raise RuntimeError(f"{name} не восстановился") from last_error

pipeline_started = time.perf_counter()
run_step("extract", extract_flaky)
run_step("validate", lambda: check_schema(raw))
run_step("load", lambda: count_second)

def build_mart_sql():
    duck("""
        CREATE OR REPLACE TABLE product_features AS
        SELECT
            asin,
            count(*) AS n_reviews,
            avg(rating) AS avg_rating,
            avg(length(text)) AS avg_text_len,
            avg(CASE WHEN verified_purchase THEN 1.0 ELSE 0.0 END) AS verified_share
        FROM reviews_clean
        GROUP BY asin
    """)
    return con.execute("SELECT count(*) FROM product_features").fetchone()[0]

mart_products = run_step("transform", build_mart_sql)
print("Товаров в витрине:", mart_products)
print(con.execute("""
    SELECT asin, n_reviews, round(avg_rating, 2) AS avg_rating
    FROM product_features
    ORDER BY n_reviews DESC
    LIMIT 5
""").df())


extract: попытка 1 не удалась (источник временно недоступен)


extract: успех с попытки 2
validate: успех с попытки 1
load: успех с попытки 1
transform: успех с попытки 1
Товаров в витрине: 115709
         asin  n_reviews  avg_rating
0  B007IAE5WY       1952        4.62
1  B00EEN2HCS       1726        4.14
2  B07C533XCW       1500        4.47
3  B00R1TAN7I       1362        4.03
4  B08L5KN7X4       1328        4.01


## Модель dbt

Тот же расчёт записан как модель dbt поверх уже загруженной таблицы. Если пакет встал, модель прогоняется официальным раннером. SQL модели лежит и в блокноте, и выполняется хранилищем.


In [5]:
proj = BASE / "dbt_project"
(proj / "models").mkdir(parents=True, exist_ok=True)
(proj / "dbt_project.yml").write_text(
    "name: lab3_reviews\n"
    "version: 1.0.0\n"
    "config-version: 2\n"
    "profile: lab3_reviews\n"
    "model-paths: [models]\n"
    "models:\n"
    "  lab3_reviews:\n"
    "    +materialized: table\n",
    encoding="utf-8",
)
(proj / "profiles.yml").write_text(
    "lab3_reviews:\n"
    "  target: dev\n"
    "  outputs:\n"
    "    dev:\n"
    "      type: duckdb\n"
    f"      path: {DB_PATH.as_posix()}\n"
    "      threads: 1\n",
    encoding="utf-8",
)
(proj / "models" / "product_features.sql").write_text(
    "select\n"
    "    asin,\n"
    "    count(*) as n_reviews,\n"
    "    avg(rating) as avg_rating,\n"
    "    avg(length(text)) as avg_text_len,\n"
    "    avg(case when verified_purchase then 1.0 else 0.0 end) as verified_share\n"
    "from {{ source('raw', 'reviews_clean') }}\n"
    "group by asin\n",
    encoding="utf-8",
)
(proj / "models" / "sources.yml").write_text(
    "version: 2\n"
    "sources:\n"
    "  - name: raw\n"
    "    schema: main\n"
    "    tables:\n"
    "      - name: reviews_clean\n",
    encoding="utf-8",
)

dbt_ok = False
con.close()
try:
    from dbt.cli.main import dbtRunner
    result = dbtRunner().invoke(["run", "--project-dir", str(proj), "--profiles-dir", str(proj)])
    dbt_ok = bool(result.success)
    print("dbt run success:", dbt_ok)
except Exception as exc:
    print("dbt не запустился, витрина уже собрана тем же SQL в DuckDB:", exc)

con = duckdb.connect(DB_PATH.as_posix())
features = con.execute("SELECT count(*) FROM product_features").fetchone()[0]
print("Строк витрины:", features)


07:36:55  Running with dbt=1.12.6


07:36:55  Registered adapter: duckdb=1.11.1


07:36:56  Unable to do partial parsing because saved manifest not found. Starting full parse.


07:36:57  Found 1 model, 1 source, 511 macros


07:36:57  


07:36:57  Concurrency: 1 threads (target='dev')


... (6 outputs left out)

07:36:57  Completed successfully


07:36:57  


07:36:57  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


dbt run success: True
Строк витрины: 115709


## Качество данных

Проверяем, что в чистом слое нет пустых товаров и оценок вне шкалы. Great Expectations подключается, если пакет установился.


In [6]:
clean = con.execute("SELECT asin, rating FROM reviews_clean LIMIT 200000").df()
null_asin = int(clean["asin"].eq("").sum())
bad_rating = int((~clean["rating"].between(1, 5)).sum())
print("Пустых asin в выборке чистого слоя:", null_asin)
print("Плохих оценок в выборке:", bad_rating)


def ge_rating_between(frame, mostly=None):
    # 0.18 умеет PandasDataset, более новый Great Expectations — через контекст
    try:
        from great_expectations.dataset import PandasDataset
        ge_df = PandasDataset(frame)
        kwargs = {"min_value": 1, "max_value": 5}
        if mostly is not None:
            kwargs["mostly"] = mostly
        result = ge_df.expect_column_values_to_be_between("rating", **kwargs)
        return bool(result["success"]), result["result"].get("unexpected_count")
    except ImportError:
        import great_expectations as gx
        context = gx.get_context(mode="ephemeral")
        source = context.data_sources.add_pandas("reviews")
        asset = source.add_dataframe_asset(name="reviews")
        batch = asset.add_batch_definition_whole_dataframe("full").get_batch(
            batch_parameters={"dataframe": frame}
        )
        kwargs = {"column": "rating", "min_value": 1, "max_value": 5}
        if mostly is not None:
            kwargs["mostly"] = mostly
        suite = gx.ExpectationSuite(
            name="rating_ok",
            expectations=[gx.expectations.ExpectColumnValuesToBeBetween(**kwargs)],
        )
        result = batch.validate(suite)
        return bool(result.success), result.results[0].result.get("unexpected_count")

ge_success = None
try:
    ge_success, _unexpected = ge_rating_between(clean.copy())
    print("Great Expectations:", ge_success)
except Exception as exc:
    print("Great Expectations не выполнился:", exc)

quality_parts = [
    1.0 if null_asin == 0 else 0.0,
    1.0 if bad_rating == 0 else 0.0,
    1.0 if ge_success else 0.5 if ge_success is None else 0.0,
    1.0 if idempotency_passed else 0.0,
]
data_quality_score = sum(quality_parts) / len(quality_parts)
pipeline_duration = time.perf_counter() - pipeline_started
print("## METRICS ##")
print(f"IDEMPOTENCY_TEST_PASSED={idempotency_passed}")
print(f"DLQ_ERROR_COUNT={len(dlq)}")
print(f"DATA_QUALITY_SCORE={data_quality_score:.2f}")
print(f"PIPELINE_DURATION_SEC={pipeline_duration:.2f}")
print(f"SCHEMA_DRIFT_CAUGHT={drift_caught}")
print(f"DBT_RAN={dbt_ok}")


Пустых asin в выборке чистого слоя: 0
Плохих оценок в выборке: 0


Great Expectations: True
## METRICS ##
IDEMPOTENCY_TEST_PASSED=True
DLQ_ERROR_COUNT=2
DATA_QUALITY_SCORE=1.00
PIPELINE_DURATION_SEC=8.95
SCHEMA_DRIFT_CAUGHT=True
DBT_RAN=True


## Результаты


In [7]:
print("Вывод")
print(
    "Собрал суточный ELT для рекомендаций по отзывам All_Beauty. "
    f"Сырых строк {len(raw)}, в чистый слой после отсечения битых попало {count_first}. "
    f"Повторная загрузка оставила те же {count_second} строк, идемпотентность {'прошла' if idempotency_passed else 'не прошла'}. "
    f"В DLQ ушло {len(dlq)} записей, искусственный дрейф схемы перехвачен: {drift_caught}. "
    f"Витрина содержит {mart_products} товаров. "
    f"Оценка качества {data_quality_score:.2f}, шаги с повтором пережили первый обрыв источника. "
    "Для этой категории DuckDB достаточен; оркестратор в блокноте заменяет Airflow, которого в Colab нет, и фиксирует те же зависимости: извлечение, проверка, загрузка, витрина."
)
con.close()


Вывод
Собрал суточный ELT для рекомендаций по отзывам All_Beauty. Сырых строк 701530, в чистый слой после отсечения битых попало 694252. Повторная загрузка оставила те же 694252 строк, идемпотентность прошла. В DLQ ушло 2 записей, искусственный дрейф схемы перехвачен: True. Витрина содержит 115709 товаров. Оценка качества 1.00, шаги с повтором пережили первый обрыв источника. Для этой категории DuckDB достаточен; оркестратор в блокноте заменяет Airflow, которого в Colab нет, и фиксирует те же зависимости: извлечение, проверка, загрузка, витрина.
